## Question 1

In [27]:
import pandas as pd
import requests
import io  

In [28]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

response = requests.get('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies', headers=headers)

tables = pd.read_html(io.StringIO(response.text)) 

In [29]:
df = tables[0]

In [30]:
df.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [31]:
years = pd.to_datetime(df['Date added']).dt.year 
counts = years.value_counts().sort_index() 
counts_recent = counts[counts.index >= 2020]

In [32]:
counts_recent

Date added
2020    10
2021    10
2022    15
2023    14
2024    16
2025    17
2026    16
Name: count, dtype: int64

## Question 2

In [33]:
import yfinance as yf                                                                                                                                                                                             
                                                                                                                                                                                                                    
data = yf.download('^GSPC', start='2026-01-01', end='2026-08-21', progress=False) 

In [34]:
data.columns

MultiIndex([( 'Close', '^GSPC'),
            (  'High', '^GSPC'),
            (   'Low', '^GSPC'),
            (  'Open', '^GSPC'),
            ('Volume', '^GSPC')],
           names=['Price', 'Ticker'])

In [35]:
close = data['Close'].squeeze()  

first = close.iloc[0]                                                                                                                                                                                             
last  = close.iloc[-1] 


In [36]:
ytd = (last / first - 1) * 100                                                                                                                                                                                    
ytd 

np.float64(11.412019253393835)

In [37]:
tickers = {                                                                                                                                                                                                       
    'US': '^GSPC',                                                                                                                                                                                                
    'China': '000001.SS',                                                                                                                                                                                         
    'Hong Kong': '^HSI',                                                                                                                                                                                          
    'Australia': '^AXJO',                                                                                                                                                                                         
    'India': '^NSEI',                                                                                                                                                                                             
    'Canada': '^GSPTSE',                                                                                                                                                                                          
    'Germany': '^GDAXI',                                                                                                                                                                                          
    'UK': '^FTSE',                                                                                                                                                                                                
    'Japan': '^N225',                                                                                                                                                                                             
    'Mexico': '^MXX',                                                                                                                                                                                             
    'Brazil': '^BVSP',                                                                                                                                                                                            
} 

In [38]:
returns = {}                                                                                                                                                                                                      
for name, tick in tickers.items():                                                                                                                                                                                
    d = yf.download(tick, start='2026-01-01', end='2026-08-21', progress=False)                                                                                                                                   
    close = d['Close'].squeeze()                                                                                                                                                                                  
    print(name, len(close), close.index[0].date(), close.index[-1].date())                                                                                                                                        
    returns[name] = (close.iloc[-1] / close.iloc[0] - 1) * 100 

US 159 2026-01-02 2026-08-20
China 153 2026-01-05 2026-08-20
Hong Kong 155 2026-01-02 2026-08-20
Australia 161 2026-01-02 2026-08-20
India 156 2026-01-01 2026-08-20
Canada 160 2026-01-02 2026-08-20
Germany 162 2026-01-02 2026-08-20
UK 161 2026-01-02 2026-08-20
Japan 154 2026-01-05 2026-08-20
Mexico 160 2026-01-02 2026-08-20
Brazil 159 2026-01-02 2026-08-20


In [39]:
ranked = pd.Series(returns).sort_values(ascending=False)                                                                                                                                                          
ranked   

Japan        27.750745
Canada       14.057466
US           11.412019
UK            8.010176
Germany       5.883203
Brazil        4.601997
Australia     4.078920
Mexico        0.324972
Hong Kong    -2.429832
China        -2.974985
India        -7.322959
dtype: float64

In [40]:
sp500 = ranked['US']                                                                                                                                                                                              
better = ranked.drop('US') > sp500                                                                                                                                                                                
better.sum()  

np.int64(2)

## Question 3

In [41]:
sp = yf.download('^GSPC', start='1950-01-01', progress=False)['Close'].squeeze()     

In [43]:
sp.cummax()  

Date
1950-01-03      16.660000
1950-01-04      16.850000
1950-01-05      16.930000
1950-01-06      16.980000
1950-01-09      17.080000
                 ...     
2026-09-22    7798.990234
2026-09-23    7798.990234
2026-09-24    7798.990234
2026-09-25    7798.990234
2026-09-28    7798.990234
Name: ^GSPC, Length: 19306, dtype: float64

In [44]:
running_max = sp.cummax()                                                                                                                                                                                         
new_ath = sp == running_max        # True on days that set a new all-time high                                                                                                                                    
new_ath.sum() 

np.int64(1537)

In [ ]:
ath_id = new_ath.cumsum()

In [46]:
groups = sp.groupby(ath_id)                                                                                                                                                                                       
lows  = groups.min()     # worst point inside each correction window                                                                                                                                              
highs = groups.max()     # the ATH that started it  

In [47]:
drawdown = (highs - lows) / highs * 100

In [50]:
drawdown.describe()

count    1537.000000
mean        1.028962
std         3.629369
min         0.000000
25%         0.000000
50%         0.000000
75%         0.533456
max        56.775388
Name: ^GSPC, dtype: float64

In [51]:
corr = drawdown[drawdown >= 5]                                                                                                                                                                                    
len(corr) 

74

In [52]:
trough_date = groups.idxmin()        # index label (date) of the min in each window                                                                                                                               
start_date  = groups.head(1).index   # first row of each window = the ATH (accepts label, no float mismatch)  

In [55]:
duration = pd.Series((trough_date - start_date).dt.days, index=highs.index)

In [56]:
mask = drawdown >= 5                                                                                                                                                                                              
print(corr.quantile([0.25, 0.5, 0.75]))                # drawdown percentiles                                                                                                                                     
print(duration[mask].quantile([0.25, 0.5, 0.75]))      # duration percentiles   

0.25     6.234677
0.50     7.986358
0.75    14.019826
Name: ^GSPC, dtype: float64
0.25    22.00
0.50    40.50
0.75    86.25
dtype: float64


In [57]:
top10 = pd.DataFrame({'drawdown': corr, 'duration': duration[mask]}).sort_values('drawdown', ascending=False).head(10)                                                                                            
top10

,drawdown,duration
^GSPC,,
1068,56.775388,517
1059,49.146948,929
555,48.203593,630
520,36.061641,543
1323,33.924960,33
732,33.509515,101
349,27.973568,196
579,27.113582,622
1414,25.425097,282


## Question 4

In [58]:
import yfinance as yf

ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)
result = ticker_obj.get_earnings_dates()

In [71]:
                                                                                                                                                                                                 
result.tail(3)

,EPS Estimate,Reported EPS,Surprise(%)
Earnings Date,,,
2021-04-29 16:00:00-04:00,0.47,0.79,67.30
2021-02-02 16:00:00-05:00,0.35,0.70,100.10
2020-10-29 16:00:00-04:00,0.38,0.62,64.25


In [64]:
prices = yf.download('AMZN', start='2020-01-01', progress=False)['Close'].squeeze()   

In [72]:
ret2 = (prices.shift(-1) / prices.shift(1) - 1) * 100 

In [73]:
result.index = result.index.normalize()      # 16:00 -> 00:00, still tz-aware                                                                                                                                     
result.index = result.index.tz_localize(None)  # drop the tz entirely 

In [74]:
merged = result.join(pd.DataFrame({'ret2': ret2}), how='left')                                                                                                                                                    
merged.notna().sum() 

EPS Estimate    25
Reported EPS    24
Surprise(%)     24
ret2            24
dtype: int64

In [75]:
mask = merged['Surprise(%)'] > 0                                                                                                                                                                                  
merged.loc[mask, 'ret2'].median()                                                                                                                                                                                 
merged[['ret2', 'Surprise(%)']].corr()  

,ret2,Surprise(%)
ret2,1.000000,0.219021
Surprise(%),0.219021,1.000000


In [76]:
mask = merged['Surprise(%)'] > 0                                                                                                                                                                                  
mask.sum()                                                                                                                                                                                                        
merged.loc[mask, 'ret2'].median() 

np.float64(0.35280649406906894)